In [ ]:
import cv2
import math
import numpy as np
import torch
from ultralytics import YOLO
import mediapipe as mp

# Phase 1: Face Mesh and Matrix Multipication
This section loads the heavy machine learning models into memory. 
By isolating this in its own cell, we prevent the system from re-initializing the models every time we restart the camera loop, saving significant compute time.
* **MediaPipe Face Mesh:** Allocated to the CPU for lightweight 468-point facial landmark tracking.

In [ ]:
def get_distance(p1, p2):
    return math.hypot(p2[0] - p1[0], p2[1] - p1[1])

In [ ]:
def calculate_ear(eye_indices, landmarks, frame_width, frame_height):
    points = [(int(landmarks.landmark[idx].x * frame_width), int(landmarks.landmark[idx].y * frame_height)) for idx in eye_indices]
    hor_dist = get_distance(points[0], points[3])
    ver_dist1 = get_distance(points[1], points[5])
    ver_dist2 = get_distance(points[2], points[4])
    if hor_dist == 0: return 0.0, points
    return (ver_dist1 + ver_dist2) / (2.0 * hor_dist), points

In [ ]:
def draw_hud_text(img, text, position, font_scale, color):
    x, y = position
    # Thick black outline with Anti-Aliasing for readability on any background
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), 5, cv2.LINE_AA) 
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, 2, cv2.LINE_AA)

LEFT_EYE = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
TRACKED_FACE_INDICES = [1, 152, 33, 263, 61, 291]

model_points = np.array([
    (0.0, 0.0, 0.0),             # Nose tip
    (0.0, -330.0, -65.0),        # Chin
    (-225.0, 170.0, -135.0),     # Left eye corner
    (225.0, 170.0, -135.0),      # Right eye corner
    (-150.0, -150.0, -125.0),    # Left mouth corner
    (150.0, -150.0, -125.0)      # Right mouth corner
], dtype=np.float32)


# phase 2:Loading of Object detection Model 
* **YOLOv8 Nano:** Allocated to the Nvidia CUDA GPU (if available) for asynchronous object detection.

In [ ]:
# MODEL INITIALIZATION
print("Loading Models into Memory...")
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(max_num_faces=2, refine_landmarks=True)

yolo_model = YOLO('yolov8n.pt')
if torch.cuda.is_available():
    yolo_model.to('cuda')
    print("CUDA is active. GPU acceleration enabled.")
else:
    print("CUDA not found. Running on CPU.")

# Phase 3: The Core Execution Engine (Live Telemetry Loop)
This is the primary `while` loop that captures the live webcam feed and processes the security state machines.

**Key Features:**
1. **Asynchronous Inference:** YOLOv8 only runs every 15 frames to prevent thermal throttling.
2. **Spatial Math (`solvePnP`):** Converts 2D facial landmarks into 3D Euler angles (Pitch/Yaw).
3. **Jitter-Proof State Machine:** Uses mathematical decay logic to filter out natural movements and hardware sensor flicker.
4. **Kill Switch:** A centralized 100-point risk score that terminates the proctoring session upon confirmed academic dishonesty.

*Press **'q'** to safely terminate the video loop without dropping the models from memory.*

In [ ]:

# CORE EXECUTION ENGINE

cap = cv2.VideoCapture(0)

risk_score = 0
frame_counter = 0
eyes_closed_frames = 0
head_turn_frames = 0
no_faces_frames = 0
multi_faces_frames = 0
current_yolo_boxes = []

while True:
    success, frame = cap.read()
    if not success: break
    h, w, _ = frame.shape
    frame_counter += 1

    # --- KILL SWITCH OVERRIDE ---
    if risk_score >= 100:
        cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 255), -1) 
        draw_hud_text(frame, "EXAM TERMINATED", (w//2 - 200, h//2), 1.5, (255, 255, 255))
        draw_hud_text(frame, "Contact Administrator", (w//2 - 150, h//2 + 40), 0.8, (255, 255, 255))
        cv2.imshow('Ultimate AI Proctoring System', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
        continue 

    # --- ASYNCHRONOUS YOLO (15-Frame Skip Logic) ---
    if frame_counter % 15 == 0:
        current_yolo_boxes = []
        results_yolo = yolo_model(frame, verbose=False)
        for r in results_yolo:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                if cls_id == 67: # Phone
                    x1, y1, x2, y2 = box.xyxy[0]
                    current_yolo_boxes.append((int(x1), int(y1), int(x2), int(y2), "Phone"))
                    risk_score = min(100, risk_score + 10) 
                elif cls_id == 73: # Book
                    x1, y1, x2, y2 = box.xyxy[0]
                    current_yolo_boxes.append((int(x1), int(y1), int(x2), int(y2), "Book"))
                    risk_score = min(100, risk_score + 10)

    for (x1, y1, x2, y2, label) in current_yolo_boxes:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
        draw_hud_text(frame, f"UNAUTHORIZED: {label}", (x1, y1 - 10), 0.6, (0, 0, 255))

    # --- MEDIAPIPE & SPATIAL GEOMETRY ---
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb_frame)

    camera_matrix = np.array([[w, 0, w / 2], [0, w, h / 2], [0, 0, 1]], dtype=np.float32)
    dist_coeffs = np.zeros((4, 1))

    # General HUD Readouts
    draw_hud_text(frame, "SYSTEM MONITOR", (20, 40), 0.7, (255, 255, 255))
    risk_color = (0, 0, 255) if risk_score > 50 else (0, 255, 0)
    draw_hud_text(frame, f"Risk Level: {risk_score}/100", (20, 80), 0.6, risk_color)

    if results.multi_face_landmarks:
        no_faces_frames = 0 
        
        if len(results.multi_face_landmarks) > 1:
            multi_faces_frames += 1
            if multi_faces_frames > 30:
                risk_score = min(100, risk_score + 50)
                multi_faces_frames = 0
            draw_hud_text(frame, "CRITICAL: MULTIPLE FACES!", (20, 240), 0.6, (0, 0, 255))
            
        else:
            multi_faces_frames = 0
            for face_landmarks in results.multi_face_landmarks:
                
                # EAR Calculation
                left_ear, _ = calculate_ear(LEFT_EYE, face_landmarks, w, h)
                right_ear, _ = calculate_ear(RIGHT_EYE, face_landmarks, w, h)
                avg_ear = (left_ear + right_ear) / 2.0
                
                if avg_ear < 0.20:
                    eyes_closed_frames += 1
                    if eyes_closed_frames > 60:
                        risk_score = min(100, risk_score + 10)
                        eyes_closed_frames = 0
                else:
                    eyes_closed_frames = 0
                
                # solvePnP Calculation
                image_points = np.array([(int(face_landmarks.landmark[idx].x * w), int(face_landmarks.landmark[idx].y * h)) for idx in TRACKED_FACE_INDICES], dtype=np.float32)
                _, rotation_vector, translation_vector = cv2.solvePnP(model_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE)
                rotation_matrix, _ = cv2.Rodrigues(rotation_vector)
                proj_matrix = np.hstack((rotation_matrix, translation_vector))
                _, _, _, _, _, _, euler_angles = cv2.decomposeProjectionMatrix(proj_matrix)
                
                pitch = euler_angles[0][0]
                yaw = euler_angles[1][0]
                
                # Print Telemetry
                draw_hud_text(frame, f"EAR: {avg_ear:.2f}", (20, 130), 0.5, (200, 200, 200))
                draw_hud_text(frame, f"Yaw: {int(yaw)} deg", (20, 160), 0.5, (200, 200, 200))
                draw_hud_text(frame, f"Pitch: {int(pitch)} deg", (20, 190), 0.5, (200, 200, 200))

                # Jitter-Proof Thresholds
                if abs(yaw) > 35 or pitch < -170:
                    head_turn_frames += 1
                    draw_hud_text(frame, "WARN: HEAD ANOMALY", (20, 240), 0.6, (0, 165, 255))
                    if head_turn_frames > 60:
                        risk_score = min(100, risk_score + 20)
                        head_turn_frames = 0
                else:
                    # Decay logic to stop millisecond flicker
                    head_turn_frames = max(0, head_turn_frames - 2)
                    if eyes_closed_frames > 15:
                        draw_hud_text(frame, "WARN: EYES CLOSED", (20, 240), 0.6, (0, 165, 255))
                    else:
                        draw_hud_text(frame, "STATUS: NOMINAL", (20, 240), 0.6, (0, 255, 0))

    else:
        no_faces_frames += 1
        if no_faces_frames > 30:
            risk_score = min(100, risk_score + 25)
            no_faces_frames = 0
        draw_hud_text(frame, "STATUS: NO FACE DETECTED", (20, 240), 0.6, (0, 0, 255))

    cv2.imshow('Ultimate AI Proctoring System', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()